In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

# Set display options
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

print("Libraries imported successfully!")

In [ ]:
# Load the data
df = pd.read_csv('LinenData.csv')
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
df.head(10)

In [ ]:
# Display basic information about the dataset
print("Dataset Info:")
print(df.info())
print("\n" + "="*50)
print("Basic Statistics:")
print(df.describe())
print("\n" + "="*50)
print("Missing Values:")
print(df.isnull().sum())

In [ ]:
# Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')

# Extract additional date features
df['Month'] = df['Date'].dt.month
df['DayOfMonth'] = df['Date'].dt.day

# Sort by date
df = df.sort_values('Date').reset_index(drop=True)

print("Date features extracted successfully!")
df.head()

In [ ]:
# Visualize BlanketUsage patterns
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Time series plot
axes[0, 0].plot(df['Date'], df['BlanketUsage'], marker='o', linestyle='-', alpha=0.7, markersize=3)
axes[0, 0].set_title('Blanket Usage Over Time', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Blanket Usage')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].grid(True, alpha=0.3)

# Distribution plot
axes[0, 1].hist(df['BlanketUsage'], bins=15, edgecolor='black', alpha=0.7, color='steelblue')
axes[0, 1].set_title('Distribution of Blanket Usage', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Blanket Usage')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Blanket Usage by Day of Week
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
avg_by_day = df.groupby('DayOfWeekNum')['BlanketUsage'].mean().reindex(range(1, 8))
axes[1, 0].bar(range(1, 8), avg_by_day, color='coral', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Average Blanket Usage by Day of Week', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Day of Week')
axes[1, 0].set_ylabel('Average Blanket Usage')
axes[1, 0].set_xticks(range(1, 8))
axes[1, 0].set_xticklabels(day_names)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Blanket Usage: Weekend vs Weekday
weekend_avg = df.groupby('IsWeekend')['BlanketUsage'].mean()
axes[1, 1].bar(['Weekday', 'Weekend'], weekend_avg, color=['lightblue', 'lightgreen'], edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Average Blanket Usage: Weekday vs Weekend', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Average Blanket Usage')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nBlanket Usage Statistics:")
print(f"Mean: {df['BlanketUsage'].mean():.2f}")
print(f"Median: {df['BlanketUsage'].median():.2f}")
print(f"Std Dev: {df['BlanketUsage'].std():.2f}")
print(f"\nAverage by Day of Week:")
for i, day in enumerate(day_names, 1):
    print(f"{day}: {avg_by_day[i]:.2f}")

## Model 1: Baseline Rule-Based Model

Simple ordering rule:
- **Friday (Day 5)**: Order 20 blankets
- **Tuesday (Day 2)**: Order 15 blankets
- **All other days**: Order 0 blankets

In [ ]:
# Define baseline model
def baseline_prediction(day_of_week):
    """
    Baseline rule:
    - Friday (5): 20
    - Tuesday (2): 15
    - Other days: 0
    """
    if day_of_week == 5:  # Friday
        return 20
    elif day_of_week == 2:  # Tuesday
        return 15
    else:
        return 0

# Apply baseline model to entire dataset
df['Baseline_Prediction'] = df['DayOfWeekNum'].apply(baseline_prediction)

print("Baseline model predictions generated!")
df[['Date', 'DayOfWeekNum', 'BlanketUsage', 'Baseline_Prediction']].head(15)

In [ ]:
# Evaluate baseline model
baseline_mae = mean_absolute_error(df['BlanketUsage'], df['Baseline_Prediction'])
baseline_rmse = np.sqrt(mean_squared_error(df['BlanketUsage'], df['Baseline_Prediction']))
baseline_r2 = r2_score(df['BlanketUsage'], df['Baseline_Prediction'])

print("="*60)
print("BASELINE MODEL PERFORMANCE (Full Dataset)")
print("="*60)
print(f"R² Score:                 {baseline_r2:.4f}")
print(f"Mean Absolute Error:      {baseline_mae:.4f}")
print(f"Root Mean Squared Error:  {baseline_rmse:.4f}")
print("="*60)

## Model 2: Random Forest Model

In [ ]:
# Prepare features and target for Random Forest
feature_columns = ['AdmCount', 'DosaCount', 'DayOfWeekNum', 'IsWeekend', 'Month', 'DayOfMonth']
X = df[feature_columns]
y = df['BlanketUsage']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {feature_columns}")

In [ ]:
# Split data into train and test sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set size:     {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

In [ ]:
# Train Random Forest model
print("Training Random Forest model...")
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
rf_model.fit(X_train, y_train)
print("✓ Model training complete!")

In [ ]:
# Make predictions on test set
y_pred_rf = rf_model.predict(X_test)

# Evaluate Random Forest model
rf_r2 = r2_score(y_test, y_pred_rf)
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print("="*60)
print("RANDOM FOREST MODEL PERFORMANCE (Test Set)")
print("="*60)
print(f"R² Score:                 {rf_r2:.4f}")
print(f"Mean Absolute Error:      {rf_mae:.4f}")
print(f"Root Mean Squared Error:  {rf_rmse:.4f}")
print("="*60)

In [ ]:
# Feature importance analysis
feature_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance)

# Visualize feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='seagreen', edgecolor='black', alpha=0.7)
plt.xlabel('Importance', fontsize=12)
plt.title('Feature Importance - Random Forest Model', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## Model Comparison

In [ ]:
# Get baseline predictions for test set
X_test_with_index = X_test.copy()
X_test_with_index['Baseline_Prediction'] = X_test_with_index['DayOfWeekNum'].apply(baseline_prediction)

# Evaluate baseline on test set
baseline_test_mae = mean_absolute_error(y_test, X_test_with_index['Baseline_Prediction'])
baseline_test_rmse = np.sqrt(mean_squared_error(y_test, X_test_with_index['Baseline_Prediction']))
baseline_test_r2 = r2_score(y_test, X_test_with_index['Baseline_Prediction'])

# Create comparison dataframe
comparison = pd.DataFrame({
    'Model': ['Baseline (Rule-Based)', 'Random Forest'],
    'R² Score': [baseline_test_r2, rf_r2],
    'MAE': [baseline_test_mae, rf_mae],
    'RMSE': [baseline_test_rmse, rf_rmse]
})

print("="*70)
print("MODEL COMPARISON (Test Set)")
print("="*70)
print(comparison.to_string(index=False))
print("="*70)

# Calculate improvement
mae_improvement = ((baseline_test_mae - rf_mae) / baseline_test_mae) * 100
rmse_improvement = ((baseline_test_rmse - rf_rmse) / baseline_test_rmse) * 100

print(f"\n📊 Random Forest Improvements:")
print(f"   MAE improvement:  {mae_improvement:+.2f}%")
print(f"   RMSE improvement: {rmse_improvement:+.2f}%")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Baseline predictions vs actual
axes[0].scatter(y_test, X_test_with_index['Baseline_Prediction'], alpha=0.6, s=50)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_title('Baseline Model: Predicted vs Actual', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Actual BlanketUsage')
axes[0].set_ylabel('Predicted BlanketUsage')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].text(0.05, 0.95, f'R² = {baseline_test_r2:.4f}\nMAE = {baseline_test_mae:.2f}', 
             transform=axes[0].transAxes, fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Random Forest predictions vs actual
axes[1].scatter(y_test, y_pred_rf, alpha=0.6, color='green', s=50)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[1].set_title('Random Forest Model: Predicted vs Actual', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Actual BlanketUsage')
axes[1].set_ylabel('Predicted BlanketUsage')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].text(0.05, 0.95, f'R² = {rf_r2:.4f}\nMAE = {rf_mae:.2f}', 
             transform=axes[1].transAxes, fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Comparison bar chart
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(comparison))
width = 0.25

ax.bar(x - width, comparison['MAE'], width, label='MAE', alpha=0.8, color='steelblue', edgecolor='black')
ax.bar(x, comparison['RMSE'], width, label='RMSE', alpha=0.8, color='coral', edgecolor='black')

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Error Value', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison['Model'])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Sample Predictions

In [ ]:
# Create sample scenarios for prediction
sample_scenarios = pd.DataFrame({
    'Scenario': ['Low admission Friday', 'High admission Friday', 'Low admission Tuesday', 'High admission Tuesday', 'Weekend day'],
    'AdmCount': [5, 15, 5, 12, 6],
    'DosaCount': [0, 3, 0, 3, 0],
    'DayOfWeekNum': [5, 5, 2, 2, 7],
    'IsWeekend': [0, 0, 0, 0, 1],
    'Month': [1, 1, 1, 1, 1],
    'DayOfMonth': [15, 16, 12, 13, 17]
})

# Get predictions from both models
sample_scenarios['Baseline_Prediction'] = sample_scenarios['DayOfWeekNum'].apply(baseline_prediction)
sample_scenarios['RF_Prediction'] = rf_model.predict(sample_scenarios[feature_columns])

print("\n" + "="*80)
print("SAMPLE PREDICTIONS")
print("="*80)
display_cols = ['Scenario', 'AdmCount', 'DosaCount', 'DayOfWeekNum', 'Baseline_Prediction', 'RF_Prediction']
print(sample_scenarios[display_cols].to_string(index=False))
print("="*80)

## Save the Best Model

In [ ]:
# Save the Random Forest model
joblib.dump(rf_model, 'blanket_usage_rf_model.pkl')
print("✓ Random Forest model saved as 'blanket_usage_rf_model.pkl'")

# Save feature columns for future predictions
joblib.dump(feature_columns, 'feature_columns.pkl')
print("✓ Feature columns saved as 'feature_columns.pkl'")

# Save model performance metrics
metrics = {
    'model_type': 'RandomForestRegressor',
    'r2_score': rf_r2,
    'mae': rf_mae,
    'rmse': rf_rmse,
    'baseline_r2': baseline_test_r2,
    'baseline_mae': baseline_test_mae,
    'baseline_rmse': baseline_test_rmse,
    'features': feature_columns
}
joblib.dump(metrics, 'model_metrics.pkl')
print("✓ Model metrics saved as 'model_metrics.pkl'")

print("\n" + "="*60)
print("All models and data saved successfully!")
print("="*60)

## Summary

### Key Findings:
- **Baseline Model**: Simple rule-based ordering (0 daily, 20 on Fridays, 15 on Tuesdays)
- **Random Forest Model**: Machine learning model using admission counts, day of week, and date features
- The Random Forest model provides more accurate predictions by learning patterns from historical data
- Feature importance analysis shows which factors most influence blanket usage

### Usage:
To use the saved model for future predictions:
```python
import joblib
model = joblib.load('blanket_usage_rf_model.pkl')
features = joblib.load('feature_columns.pkl')
# Make predictions with new data
```